In [ ]:
!pip install kafka-python-ng pyspark

import sys
from time import sleep
from json import dumps
from kafka import KafkaProducer
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

print("=== INICIANDO PIPELINE DATAOPS ===")

# ==========================================
# ETAPA 1: INGESTA (Simulación con Kafka)
# ==========================================
try:
    producer = KafkaProducer(
        bootstrap_servers=['localhost:9092'],
        value_serializer=lambda x: dumps(x).encode('utf-8')
    )
    # Mandamos un mensaje de prueba al tópico 'examen_topic'
    dato_simulado = {"id": 1, "nombre": "Cliente_Duoc", "edad": 25}
    producer.send('examen_topic', value=dato_simulado)
    producer.flush()
    print("[OK] Etapa 1: Ingesta en Kafka exitosa.")
except Exception as e:
    print(f"[ERROR] Falló Ingesta Kafka: {e}")

# ==========================================
# ETAPA 2 Y 3: PROCESAMIENTO Y VALIDACIÓN (Spark)
# ==========================================
try:
    # Conexión al Spark Master que tienes corriendo localmente
    spark = SparkSession.builder \
        .appName("DataOpsPipelineExamen") \
        .master("spark://localhost:7077") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
        .getOrCreate()

    # Estructura/Esquema estricto (Validación Estructural)
    schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("nombre", StringType(), True),
        StructField("edad", IntegerType(), True)
    ])

    # Convertimos nuestro dato simulado a DataFrame de Spark
    df = spark.createDataFrame([dato_simulado], schema)

    # Simulación de Limpieza: Filtrar filas donde la edad sea inválida (Semántica)
    df_limpio = df.filter(df.edad > 0)
    print("[OK] Etapa 2 y 3: Limpieza y Validación con Spark procesada.")
except Exception as e:
    print(f"[ERROR] Falló Procesamiento Spark: {e}")

# ==========================================
# ETAPA 4: CARGA (Guardar en PostgreSQL)
# ==========================================
try:
    # Escribir el DataFrame procesado en la tabla de Postgres
    df_limpio.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://localhost:5431/db_examen") \
        .option("dbtable", "reporte_pipeline") \
        .option("user", "postgres") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()
    print("[OK] Etapa 4: Carga en PostgreSQL db_examen completada.")
except Exception as e:
    print(f"[ERROR] Falló Carga Postgres: {e}")

print("=== PIPELINE FINALIZADO ===")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 5.0 MB/s eta 0:00:00


ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. K

=== INICIANDO PIPELINE DATAOPS ===
[ERROR] Falló Ingesta Kafka: NoBrokersAvailable
[ERROR] Falló Procesamiento Spark: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.jav